# Token-Level Ablation

Per-token causal analysis: knockout effects, necessity/sufficiency testing,
minimal token set identification, pairwise interaction measurement, and
comprehensive importance ranking.

## Why This Matters

Ablation token level measures the effect of removing or zeroing specific components. Ablation is the simplest causal intervention: if removing a component changes behavior, that component is necessary. Systematic ablation reveals the importance hierarchy of model components.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_level_ablation import (
    per_token_knockout,
    token_necessity_sufficiency,
    minimal_token_set,
    pairwise_token_interaction,
    token_importance_ranking,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Per-Token Knockout

In [ ]:
result = per_token_knockout(model, tokens, metric_fn)
print(f"Base metric: {result['base_metric']:.4f}")
print(f"\nToken effects:")
for pos, effect in result['most_important_positions']:
    print(f"  Position {pos} (token {int(tokens[pos])}): effect={effect:+.4f}")

## Necessity and Sufficiency

In [ ]:
result = token_necessity_sufficiency(model, tokens, metric_fn)
print(f"Correlation: {result['necessity_sufficiency_correlation']:.4f}")
for pos in range(len(tokens)):
    print(f"  Pos {pos}: necessity={result['necessity'][pos]:.3f}, "
          f"sufficiency={result['sufficiency'][pos]:.3f}")

## Minimal Token Set

In [ ]:
result = minimal_token_set(model, tokens, metric_fn, threshold=0.8)
print(f"Minimal set: {result['minimal_set']} ({result['set_size']} tokens)")
print(f"Coverage: {result['coverage']:.1%}")
print(f"\nTrajectory:")
for pos, metric in result['metric_trajectory']:
    print(f"  +pos {pos}: metric={metric:.4f}")

## Pairwise Token Interaction

In [ ]:
result = pairwise_token_interaction(model, tokens, metric_fn)
print(f"Max interaction: pos {result['max_interaction'][0]}-{result['max_interaction'][1]}: {result['max_interaction'][2]:.4f}")
if result['synergistic_pairs']:
    print("\nSynergistic pairs:")
    for i, j, v in result['synergistic_pairs'][:3]:
        print(f"  pos {i}-{j}: synergy={v:.4f}")
if result['redundant_pairs']:
    print("\nRedundant pairs:")
    for i, j, v in result['redundant_pairs'][:3]:
        print(f"  pos {i}-{j}: redundancy={v:.4f}")

## Token Importance Ranking

In [ ]:
result = token_importance_ranking(model, tokens, metric_fn)
print(f"Entropy: {result['entropy']:.4f}")
print(f"\nRanking:")
for pos, score in result['ranking']:
    print(f"  Position {pos} (token {int(tokens[pos])}): {score:.4f} ({result['normalized_scores'][pos]:.1%})")